In [1]:
import pandas as pd

In [5]:
import pandas as pd
import numpy as np


def create_multivariate_windows(
    df,
    window_size=24,
    forecast_horizon=24,
    feature_cols=None,
    stride=1,
):
    """
    Create 2D sliding windows for multivariate time series modeling.
`
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with columns: site, date, start_hour, and feature columns
    window_size : int
        Size of the sliding window (default: 24 for hours in a day)
    forecast_horizon : int
        Number of time steps to predict ahead (default: 2)
    feature_cols : list
        List of column names to include as features
    stride : int
        Step size for sliding the window (default: 1)

    Returns:
    --------
    X : np.ndarray
        Array of shape (n_windows, window_size, n_features) containing input sequences
    y : np.ndarray
        Array of shape (n_windows, forecast_horizon, n_features) containing target sequences
    metadata : pd.DataFrame
        Dataframe containing metadata for each window
    """

    if feature_cols is None:
        feature_cols = []
    # Verify all feature columns exist
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    # Sort data by site, date, and start_hour
    df = df.sort_values(["site", "date", "start_hour"]).reset_index(drop=True)

    X = []
    y = []
    metadata = []

    # Process each site separately to ensure no cross-site windows
    for site in df["site"].unique():
        site_data = df[df["site"] == site].reset_index(drop=True)

        # Calculate the maximum valid starting index
        # We need window_size + forecast_horizon rows to create a valid sample
        max_start_idx = len(site_data) - window_size - forecast_horizon + 1

        # Create sliding windows with specified stride
        for i in range(0, max_start_idx, stride):
            # Extract input window (window_size × n_features)
            window_end_idx = i + window_size
            X_window = site_data[feature_cols].iloc[i:window_end_idx].values

            # Extract target window (forecast_horizon × n_features)
            target_start_idx = window_end_idx
            target_end_idx = target_start_idx + forecast_horizon
            y_window = (
                site_data[feature_cols].iloc[target_start_idx:target_end_idx].values
            )

            # Verify we got the correct shapes
            if (
                X_window.shape[0] == window_size
                and y_window.shape[0] == forecast_horizon
            ):
                X.append(X_window)
                y.append(y_window)

                # Store metadata
                metadata.append(
                    {
                        "site": site,
                        "window_start_idx": i,
                        "X_start_date": site_data["date"].iloc[i],
                        "X_start_hour": site_data["start_hour"].iloc[i],
                        "X_end_date": site_data["date"].iloc[window_end_idx - 1],
                        "X_end_hour": site_data["start_hour"].iloc[window_end_idx - 1],
                        "y_start_date": site_data["date"].iloc[target_start_idx],
                        "y_start_hour": site_data["start_hour"].iloc[target_start_idx],
                        "y_end_date": site_data["date"].iloc[target_end_idx - 1],
                        "y_end_hour": site_data["start_hour"].iloc[target_end_idx - 1],
                    }
                )

    # Convert to numpy arrays
    X = np.array(X)  # Shape: (n_samples, window_size, n_features)
    y = np.array(y)  # Shape: (n_samples, forecast_horizon, n_features)
    metadata_df = pd.DataFrame(metadata)

    return X, y, metadata_df


def split_train_test_by_site(X, y, metadata, test_size=0.2, random_state=42):
    """
    Split data into train and test sets, keeping all windows from each site together.

    Parameters:
    -----------
    X : np.ndarray
        Input windows
    y : np.ndarray
        Target windows
    metadata : pd.DataFrame
        Metadata with site information
    test_size : float
        Proportion of sites to use for testing
    random_state : int
        Random seed for reproducibility

    Returns:
    --------
    X_train, X_test, y_train, y_test, metadata_train, metadata_test
    """
    np.random.seed(random_state)

    # Get unique sites
    unique_sites = metadata["site"].unique()
    n_test_sites = max(1, int(len(unique_sites) * test_size))

    # Randomly select test sites
    test_sites = np.random.choice(unique_sites, size=n_test_sites, replace=False)

    # Create train/test masks
    test_mask = metadata["site"].isin(test_sites)
    train_mask = ~test_mask

    return (
        X[train_mask],
        X[test_mask],
        y[train_mask],
        y[test_mask],
        metadata[train_mask].reset_index(drop=True),
        metadata[test_mask].reset_index(drop=True),
    )


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime


class PM25Dataset(Dataset):
    """PyTorch Dataset for PM2.5 time series data."""

    def __init__(self, X, y):
        """
        Args:
            X: np.ndarray of shape (n_samples, window_size, n_features)
            y: np.ndarray of shape (n_samples, forecast_horizon, n_features)
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class PM25LSTM(nn.Module):
    """
    LSTM model for PM2.5 prediction.

    Architecture:
    - Multi-layer LSTM for temporal pattern learning
    - Dropout for regularization
    - Fully connected layer for output projection
    """

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2,
        dropout=0.2,
        forecast_horizon=24,
        output_size=1
    ):
        super(PM25LSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.forecast_horizon = forecast_horizon

        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Fully connected layer to project LSTM output to forecast horizon
        self.fc = nn.Linear(hidden_size, forecast_horizon * output_size)

        self.output_size = output_size

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch_size, window_size, input_size)

        Returns:
            Output tensor of shape (batch_size, forecast_horizon, output_size)
        """
        # LSTM forward pass
        # lstm_out shape: (batch_size, window_size, hidden_size)
        lstm_out, (hidden, cell) = self.lstm(x)

        # Take the last output
        last_output = lstm_out[:, -1, :]  # (batch_size, hidden_size)

        # Apply dropout
        last_output = self.dropout(last_output)

        # Project to forecast horizon
        output = self.fc(last_output)  # (batch_size, forecast_horizon * output_size)

        # Reshape to (batch_size, forecast_horizon, output_size)
        output = output.view(-1, self.forecast_horizon, self.output_size)

        return output


class EarlyStopping:
    """Early stopping to prevent overfitting."""

    def __init__(self, patience=10, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict().copy()
            self.counter = 0

        return self.early_stop


def train_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        optimizer.zero_grad()
        y_pred = model(X_batch)

        # Compute loss
        loss = criterion(y_pred, y_batch)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_model(model, dataloader, scaler, device):
    """
    Evaluate model performance with multiple metrics.

    Returns:
        dict: Dictionary containing evaluation metrics
    """
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)

            all_preds.append(y_pred.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    # Concatenate all predictions and targets
    y_pred = np.concatenate(all_preds, axis=0)  # (n_samples, forecast_horizon, n_features)
    y_true = np.concatenate(all_targets, axis=0)

    # Inverse transform if scaler is provided
    if scaler is not None:
        # Reshape for inverse transform
        original_shape = y_pred.shape
        y_pred_2d = y_pred.reshape(-1, y_pred.shape[-1])
        y_true_2d = y_true.reshape(-1, y_true.shape[-1])

        y_pred_original = scaler.inverse_transform(y_pred_2d)
        y_true_original = scaler.inverse_transform(y_true_2d)

        y_pred = y_pred_original.reshape(original_shape)
        y_true = y_true_original.reshape(original_shape)

    # Calculate metrics
    mse = np.mean((y_pred - y_true) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_pred - y_true))

    # R² score
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    # MAPE (Mean Absolute Percentage Error) - avoid division by zero
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    metrics = {
        'rmse': float(rmse),
        'mae': float(mae),
        'mse': float(mse),
        'r2': float(r2),
        'mape': float(mape)
    }

    return metrics, y_pred, y_true


def plot_predictions(y_true, y_pred, n_samples=5, save_path=None):
    """Plot sample predictions vs actual values."""
    fig, axes = plt.subplots(n_samples, 1, figsize=(12, 3 * n_samples))

    if n_samples == 1:
        axes = [axes]

    for i in range(min(n_samples, len(y_true))):
        axes[i].plot(y_true[i, :, 0], label='Actual', marker='o', linestyle='-', alpha=0.7)
        axes[i].plot(y_pred[i, :, 0], label='Predicted', marker='x', linestyle='--', alpha=0.7)
        axes[i].set_xlabel('Hour')
        axes[i].set_ylabel('PM2.5')
        axes[i].set_title(f'Sample {i+1}')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Predictions plot saved to {save_path}")

    plt.close()




# ============================================================================
# Configuration
# ============================================================================
CONFIG = {
    # Data parameters
    'data_path': '/Users/jiangli/Desktop/wildfire-air-quality/data/final_merged_data.csv',
    'feature_cols': ['pm25'],
    'window_size': 24,  # 24 hours of historical data
    'forecast_horizon': 24,  # Predict next 24 hours
    'stride': 1,
    'test_size': 0.2,
    'random_state': 42,

    # Model parameters
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.2,

    # Training parameters
    'batch_size': 32,
    'learning_rate': 0.001,
    'num_epochs': 100,
    'early_stopping_patience': 15,

    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # Output paths
    'output_dir': '../models',
    'model_name': 'pm25_lstm'
}

print("=" * 80)
print("PM2.5 Prediction Model Training")
print("=" * 80)
print(f"\nDevice: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

# Create output directory
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)

# ============================================================================
# Load and prepare data
# ============================================================================
print("Loading data...")
df = pd.read_csv(CONFIG['data_path'])
print(f"Loaded {len(df)} records from {df['site'].nunique()} sites")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")

# Check for missing PM2.5 values
missing_pm25 = df['pm25'].isna().sum()
print(f"\nMissing PM2.5 values: {missing_pm25} ({missing_pm25/len(df)*100:.2f}%)")

# Drop rows with missing PM2.5
if missing_pm25 > 0:
    df = df.dropna(subset=['pm25'])
    print(f"After dropping missing values: {len(df)} records")

# ============================================================================
# Create sliding windows
# ============================================================================
print("\n" + "=" * 80)
print("Creating sliding windows...")
print("=" * 80)

X, y, metadata = create_multivariate_windows(
    df,
    window_size=CONFIG['window_size'],
    forecast_horizon=CONFIG['forecast_horizon'],
    feature_cols=CONFIG['feature_cols'],
    stride=CONFIG['stride']
)

print(f"\nInput (X) shape: {X.shape}")
print(f"  - {X.shape[0]} samples")
print(f"  - {X.shape[1]} time steps")
print(f"  - {X.shape[2]} features")

print(f"\nTarget (y) shape: {y.shape}")
print(f"  - {y.shape[0]} samples")
print(f"  - {y.shape[1]} forecast horizon")
print(f"  - {y.shape[2]} features")

# ============================================================================
# Normalize data
# ============================================================================
print("\n" + "=" * 80)
print("Normalizing data...")
print("=" * 80)

# Fit scaler on training data only (we'll split first)
# But for now, let's just note the original statistics
print(f"PM2.5 statistics (before normalization):")
print(f"  Mean: {X[:, :, 0].mean():.2f}")
print(f"  Std: {X[:, :, 0].std():.2f}")
print(f"  Min: {X[:, :, 0].min():.2f}")
print(f"  Max: {X[:, :, 0].max():.2f}")

# ============================================================================
# Split data
# ============================================================================
print("\n" + "=" * 80)
print("Splitting data into train/test sets...")
print("=" * 80)

X_train, X_test, y_train, y_test, meta_train, meta_test = split_train_test_by_site(
    X, y, metadata,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state']
)

print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Sites: {sorted(meta_train['site'].unique())}")

print(f"\nTest set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Sites: {sorted(meta_test['site'].unique())}")

# ============================================================================
# Normalize using training data statistics
# ============================================================================
scaler = StandardScaler()

# Reshape for scaling: (n_samples * time_steps, n_features)
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_test_2d = X_test.reshape(-1, X_test.shape[-1])
y_train_2d = y_train.reshape(-1, y_train.shape[-1])
y_test_2d = y_test.reshape(-1, y_test.shape[-1])

# Fit on training data
scaler.fit(X_train_2d)

# Transform both X and y
X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)
y_train_scaled = scaler.transform(y_train_2d).reshape(y_train.shape)
y_test_scaled = scaler.transform(y_test_2d).reshape(y_test.shape)

print(f"\nAfter normalization:")
print(f"  Mean: {X_train_scaled.mean():.4f}")
print(f"  Std: {X_train_scaled.std():.4f}")

# ============================================================================
# Create PyTorch datasets and dataloaders
# ============================================================================
print("\n" + "=" * 80)
print("Creating PyTorch datasets...")
print("=" * 80)

train_dataset = PM25Dataset(X_train_scaled, y_train_scaled)
test_dataset = PM25Dataset(X_test_scaled, y_test_scaled)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0  # Set to 0 for Windows compatibility
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# ============================================================================
# Initialize model
# ============================================================================
print("\n" + "=" * 80)
print("Initializing model...")
print("=" * 80)

device = torch.device(CONFIG['device'])

model = PM25LSTM(
    input_size=len(CONFIG['feature_cols']),
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    forecast_horizon=CONFIG['forecast_horizon'],
    output_size=len(CONFIG['feature_cols'])
).to(device)

print(f"\nModel architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ============================================================================
# Training setup
# ============================================================================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)
early_stopping = EarlyStopping(patience=CONFIG['early_stopping_patience'])

# ============================================================================
# Training loop
# ============================================================================
print("\n" + "=" * 80)
print("Training model...")
print("=" * 80)

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(CONFIG['num_epochs']):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)

    # Validate
    val_loss = validate(model, test_loader, criterion, device)
    val_losses.append(val_loss)

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}] "
            f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': CONFIG
        }, output_dir / f"{CONFIG['model_name']}_best.pth")
        print(f"  → Saved best model (val_loss: {val_loss:.6f})")

    # Early stopping
    if early_stopping(val_loss, model):
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        # Restore best model
        model.load_state_dict(early_stopping.best_model_state)
        break

# ============================================================================
# Plot training history
# ============================================================================
print("\n" + "=" * 80)
print("Plotting training history...")
print("=" * 80)

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', alpha=0.7)
plt.plot(val_losses, label='Validation Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.savefig(output_dir / f"{CONFIG['model_name']}_training_history.png", dpi=300)
print(f"Training history saved to {output_dir / f'{CONFIG['model_name']}_training_history.png'}")
plt.close()

# ============================================================================
# Evaluate model
# ============================================================================
print("\n" + "=" * 80)
print("Evaluating model...")
print("=" * 80)

# Load best model
checkpoint = torch.load(output_dir / f"{CONFIG['model_name']}_best.pth")
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate on test set
test_metrics, y_pred, y_true = evaluate_model(model, test_loader, scaler, device)

print("\nTest Set Performance:")
print(f"  RMSE: {test_metrics['rmse']:.4f}")
print(f"  MAE:  {test_metrics['mae']:.4f}")
print(f"  MSE:  {test_metrics['mse']:.4f}")
print(f"  R²:   {test_metrics['r2']:.4f}")
print(f"  MAPE: {test_metrics['mape']:.2f}%")

# Evaluate on training set for comparison
train_metrics, _, _ = evaluate_model(model, train_loader, scaler, device)

print("\nTraining Set Performance:")
print(f"  RMSE: {train_metrics['rmse']:.4f}")
print(f"  MAE:  {train_metrics['mae']:.4f}")
print(f"  R²:   {train_metrics['r2']:.4f}")

# ============================================================================
# Plot sample predictions
# ============================================================================
print("\n" + "=" * 80)
print("Plotting sample predictions...")
print("=" * 80)

plot_predictions(
    y_true, y_pred,
    n_samples=5,
    save_path=output_dir / f"{CONFIG['model_name']}_predictions.png"
)

# ============================================================================
# Save scaler and final artifacts
# ============================================================================
print("\n" + "=" * 80)
print("Saving artifacts...")
print("=" * 80)

# Save scaler
import joblib
joblib.dump(scaler, output_dir / f"{CONFIG['model_name']}_scaler.pkl")
print(f"Scaler saved to {output_dir / f'{CONFIG['model_name']}_scaler.pkl'}")

# Save configuration and metrics
results = {
    'config': CONFIG,
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'training_history': {
        'train_losses': [float(x) for x in train_losses],
        'val_losses': [float(x) for x in val_losses]
    },
    'timestamp': datetime.now().isoformat()
}

with open(output_dir / f"{CONFIG['model_name']}_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_dir / f'{CONFIG['model_name']}_results.json'}")

print("\n" + "=" * 80)
print("Training completed successfully!")
print("=" * 80)
print(f"\nModel files saved in: {output_dir}")
print(f"  - {CONFIG['model_name']}_best.pth (model checkpoint)")
print(f"  - {CONFIG['model_name']}_scaler.pkl (data scaler)")
print(f"  - {CONFIG['model_name']}_results.json (metrics and config)")
print(f"  - {CONFIG['model_name']}_training_history.png (loss curves)")
print(f"  - {CONFIG['model_name']}_predictions.png (sample predictions)")


PM2.5 Prediction Model Training

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/jiangli/Desktop/wildfire-air-quality/data/final_merged_data.csv'